## Reading from Bronze layer

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

## Data Transformations

In [0]:
df.display()

In [0]:
## trimming the extra whitespace in all strings 
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col
from pyspark.sql.types import StringType

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

In [0]:
## changing column names 
mapping = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key', 
    'cst_firstname':'first_name', 
    'cst_lastname':'last_name',
    'cst_marital_status':'marital_status', 
    'cst_gndr':'gender',
    'cst_create_date':'customer_create_date'
    }
df1 = df.withColumnsRenamed(mapping)
df1.display()

In [0]:
## Replacing short form for full form in gender and marital status column 
df2 = df1.replace({'M':'Male', 'F':'Female'}, subset=['gender'])
df3= df2.replace({'S':'Single', 'M':'Married'}, subset=['marital_status'])
df3.display()

In [0]:
## check for duplicates, if no duplicate values then write to delta table
duplicate_count = df3.groupBy(df3.columns[0]).count().filter("count > 1")
display(duplicate_count)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Remove entries where customer_id is null
df_cleaned = df3.filter(F.col("customer_id").isNotNull())

# 2. Calculate the number of null values in each row
# We sum the result of an 'isNull' check for every column in the row
null_count_col = sum([F.col(c).isNull().cast("int") for c in df_cleaned.columns])
df_with_null_metrics = df_cleaned.withColumn("null_count", null_count_col)

# 3. Define a Window to identify the "best" record per customer_id
# We order by null_count (lowest first)
window_spec = Window.partitionBy("customer_id").orderBy(F.col("null_count").asc())

# 4. Filter to keep only the top-ranked row (rank 1)
df_final = (df_with_null_metrics
            .withColumn("rank", F.row_number().over(window_spec))
            .filter(F.col("rank") == 1)
            .drop("null_count", "rank"))

# Display results
df_final.display()

In [0]:
duplicate_count = df_final.groupBy(df_final.columns[0]).count().filter("count > 1")
display(duplicate_count)

## Write into Silver Table

In [0]:
df_final.write.mode("overwrite").format('delta').saveAsTable("workspace.silver.crm_customer_info_cleaned")